# Cross-Reference Analysis: LLM Responses without XAI

Compares LLM outputs across sample sizes (10, 20, 40) to identify trends and diminishing returns.

In [6]:
import json
import re
from collections import defaultdict
import pandas as pd

GROUND_TRUTH = {
    'BotAttack': ['Port', 'Status', 'Payload_Size'],
    'Normal': ['Port', 'Status', 'Payload_Size'],
    'PortScan': ['Payload_Size', 'Status', 'Port'],
}

print("Ground Truth:", GROUND_TRUTH)

Ground Truth: {'BotAttack': ['Port', 'Status', 'Payload_Size'], 'Normal': ['Port', 'Status', 'Payload_Size'], 'PortScan': ['Payload_Size', 'Status', 'Port']}


In [7]:
def extract_feature_rankings(text):
    features = ['Port', 'Status', 'Payload_Size', 'Request_Type', 'Protocol', 'User_Agent']
    feature_mentions = defaultdict(int)
    
    for feat in features:
        importance_contexts = ['important', 'significant', 'key', 'main', 'primary', 'drives', 'influences', 'impact']
        for ctx in importance_contexts:
            pattern = feat + r'[^.]*' + ctx + r'[^.]|' + ctx + r'[^.]' + feat
            if re.search(pattern, text, re.IGNORECASE):
                feature_mentions[feat] += 1
    
    return dict(feature_mentions)


def validate_response(model_name, text):
    mentions = extract_feature_rankings(text)
    top3 = sorted(mentions.items(), key=lambda x: -x[1])[:3]
    top3_features = [f[0] for f in top3]
    
    user_agent_overvalued = mentions.get("User_Agent", 0) > 0
    has_port_top = mentions.get("Port", 0) == max(mentions.values()) if mentions else False
    has_correct_top3 = 'Port' in top3_features and 'Status' in top3_features
    
    return {
        'model': model_name,
        'char_count': len(text),
        'feature_mentions': mentions,
        'top3_features': top3_features,
        'user_agent_overvalued': user_agent_overvalued,
        'has_port_top': has_port_top,
        'has_correct_top3': has_correct_top3
    }

In [8]:
SAMPLE_SIZES = [10, 20, 40]
all_data = {}
all_validations = {}

for n in SAMPLE_SIZES:
    filename = f'resultados_without_xai_samples_{n}.json'
    with open(filename, 'r', encoding='utf-8') as f:
        all_data[n] = json.load(f)
    all_validations[n] = {}
    for model, text in all_data[n].items():
        all_validations[n][model] = validate_response(model, text)
    print(f"Loaded {filename}: {len(all_data[n])} models")

Loaded resultados_without_xai_samples_10.json: 4 models
Loaded resultados_without_xai_samples_20.json: 4 models
Loaded resultados_without_xai_samples_40.json: 4 models


In [9]:
cross_ref_data = []

for n in SAMPLE_SIZES:
    for model in all_validations[n].keys():
        val = all_validations[n][model]
        cross_ref_data.append({
            'N_SAMPLES': n,
            'Model': model,
            'Response Length': val['char_count'],
            'Port as Top': val['has_port_top'],
            'Correct Top-3': val['has_correct_top3'],
            'User_Agent Bias': val['user_agent_overvalued'],
        })

df_cross = pd.DataFrame(cross_ref_data)
print("\nCross-Reference Summary:")
print(df_cross.to_string(index=False))


Cross-Reference Summary:
 N_SAMPLES                Model  Response Length  Port as Top  Correct Top-3  User_Agent Bias
        10 glm-4.7-flash:latest             6533        False          False            False
        10            qwen3:14b             5915        False          False            False
        10          gpt-oss:20b             8765         True          False            False
        10            qwen3:30b             6977         True          False            False
        20 glm-4.7-flash:latest             7306         True          False             True
        20            qwen3:14b             8163         True           True             True
        20          gpt-oss:20b            11934         True          False             True
        20            qwen3:30b             6249        False          False            False
        40 glm-4.7-flash:latest             7141        False          False            False
        40            qwen3:14b   

In [10]:
print("\n" + "="*70)
print("TREND ANALYSIS")
print("="*70)

for n in SAMPLE_SIZES:
    vals = list(all_validations[n].values())
    acc = sum(1 for v in vals if v['has_correct_top3']) / len(vals) * 100
    ua = sum(1 for v in vals if v['user_agent_overvalued']) / len(vals) * 100
    avg_len = sum(v['char_count'] for v in vals) / len(vals)
    print(f"N={n}: Accuracy={acc:.0f}%, UA-Bias={ua:.0f}%, AvgLen={avg_len:.0f}")


TREND ANALYSIS
N=10: Accuracy=0%, UA-Bias=0%, AvgLen=7048
N=20: Accuracy=25%, UA-Bias=75%, AvgLen=8413
N=40: Accuracy=25%, UA-Bias=50%, AvgLen=7978


In [11]:
print("\n" + "="*70)
print("MODEL-SPECIFIC TRENDS")
print("="*70)

models = list(all_data[10].keys())

for model in models:
    print(f"\n{model}:")
    for n in SAMPLE_SIZES:
        val = all_validations[n][model]
        top3 = val['top3_features']
        ua = "YES" if val['user_agent_overvalued'] else "no"
        correct = "OK" if val['has_correct_top3'] else "miss"
        print(f"  N={n}: Top3={top3} | UA={ua} | {correct}")


MODEL-SPECIFIC TRENDS

glm-4.7-flash:latest:
  N=10: Top3=[] | UA=no | miss
  N=20: Top3=['Port', 'User_Agent'] | UA=YES | miss
  N=40: Top3=['Payload_Size'] | UA=no | miss

qwen3:14b:
  N=10: Top3=[] | UA=no | miss
  N=20: Top3=['Port', 'Status', 'Payload_Size'] | UA=YES | OK
  N=40: Top3=['User_Agent'] | UA=YES | miss

gpt-oss:20b:
  N=10: Top3=['Port'] | UA=no | miss
  N=20: Top3=['Port', 'User_Agent'] | UA=YES | miss
  N=40: Top3=[] | UA=no | miss

qwen3:30b:
  N=10: Top3=['Port'] | UA=no | miss
  N=20: Top3=['Payload_Size'] | UA=no | miss
  N=40: Top3=['Port', 'Status', 'User_Agent'] | UA=YES | OK


In [12]:
print("\n" + "="*70)
print("CONCLUSION")
print("="*70)

acc_10 = sum(1 for v in all_validations[10].values() if v['has_correct_top3']) / 4 * 100
acc_20 = sum(1 for v in all_validations[20].values() if v['has_correct_top3']) / 4 * 100
acc_40 = sum(1 for v in all_validations[40].values() if v['has_correct_top3']) / 4 * 100

print(f"\nAccuracy trend: N=10: {acc_10:.0f}% -> N=20: {acc_20:.0f}% -> N=40: {acc_40:.0f}%")

if acc_20 >= acc_10 and acc_40 >= acc_20:
    print("-> Monotonic improvement with more data")
elif acc_20 > acc_10 and acc_40 <= acc_20:
    print("-> Diminishing returns: 20 samples may be optimal")
else:
    print("-> Mixed results - no clear trend")


CONCLUSION

Accuracy trend: N=10: 0% -> N=20: 25% -> N=40: 25%
-> Monotonic improvement with more data
